# Phase B — Neighbourhood KPI Aggregation

**Owner:** Member 2 (Mohammed)

Aggregates listing-level features into one row per (city, geo_key). The output is the single table Member 3 will cluster on (adding `cluster_label` and `risk_priority_score`) and Member 4's chatbot will query.

## Design

Aggregation logic lives in `src/kpis.py` — pure function, no IO. This notebook chains directly off Phase A's output (`*_listings_features.csv`) — no recomputation, no schema drift.

## Output

- `data/processed/neighbourhood_kpis.csv` — both cities stacked, ready for clustering
- `data/processed/data_dictionary_neighbourhood_kpis.csv` — column reference for M3/M4

## 1. Imports & path setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np

from src.data_io import PROCESSED_DIR
from src.kpis import compute_neighbourhood_kpis, kpi_data_dictionary

print('Repo root:', REPO_ROOT)

Repo root: C:\Users\moham\Desktop\Term 3\Capstone\Repo\KPMG_Airbnb_Capstone


## 2. Load Phase A features for both cities and stack

Phase A guaranteed identical schema between BCN and LDN, so `pd.concat` is safe.

In [2]:
bcn = pd.read_csv(PROCESSED_DIR / 'barcelona' / 'barcelona_listings_features.csv')
ldn = pd.read_csv(PROCESSED_DIR / 'london' / 'london_listings_features.csv')

assert set(bcn.columns) == set(ldn.columns), 'Schema drift between cities!'

features = pd.concat([bcn, ldn], ignore_index=True)
print(f'BCN listings   : {len(bcn):,}')
print(f'LDN listings   : {len(ldn):,}')
print(f'Combined shape : {features.shape}')
print('city counts    :')
print(features["city"].value_counts())

BCN listings   : 2,594
LDN listings   : 9,643
Combined shape : (12237, 50)
city counts    :
city
london       9643
barcelona    2594
Name: count, dtype: int64


## 3. Compute neighbourhood KPIs

One row per (city, geo_key, geo_level). Listings with no known geo are dropped.

In [3]:
kpis = compute_neighbourhood_kpis(features)
print(f'KPI table shape: {kpis.shape}')
print(f'Neighbourhoods per city:')
print(kpis["city"].value_counts())
print(f'\nGeo-level mix:')
print(kpis.groupby(["city", "geo_level"]).size())

KPI table shape: (560, 27)
Neighbourhoods per city:
city
london       494
barcelona     66
Name: count, dtype: int64

Geo-level mix:
city       geo_level   
barcelona  subdivision      66
london     neighborhood     39
           subdivision     455
dtype: int64


In [4]:
kpis.head(10)

,city,geo_key,geo_level,str_density,active_listings,entire_home_count,median_nightly_price,p25_price,p75_price,avg_occupancy,...,entire_home_share,multi_listing_host_share,active_share,breach_rate_90,breach_rate_60,breach_rate_30,reside_unregistered_share,professional_management_share,tier_concentration_price,tier_sample_adequate
0,barcelona,Can Baró,subdivision,7,6,4,81.60,72.250,153.750,0.1951,...,0.5714,0.4286,0.8571,0.5000,0.7500,1.0000,0.0000,0.1667,tier_3,True
1,barcelona,Diagonal Mar i el Front Marítim del Poblenou,subdivision,19,10,13,215.70,171.900,323.900,0.1050,...,0.6842,0.4211,0.5263,0.3077,0.3846,0.3846,0.4615,0.1250,tier_2,True
2,barcelona,Gothic Quarter,subdivision,160,90,71,115.30,72.450,205.150,0.1091,...,0.4438,0.4688,0.5625,0.2535,0.3239,0.3662,0.4366,0.1061,tier_2,True
3,barcelona,Horta,subdivision,1,0,0,0.00,0.000,0.000,0.0000,...,0.0000,0.0000,0.0000,NaN,NaN,NaN,NaN,0.0000,tier_3,False
4,barcelona,Hostafrancs,subdivision,32,16,14,93.10,57.900,183.050,0.1913,...,0.4375,0.4688,0.5000,0.2857,0.3571,0.3571,0.5000,0.0000,tier_3,True
5,barcelona,Montbau,subdivision,2,0,1,NaN,NaN,NaN,0.0000,...,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,NaN,tier_3,False
6,barcelona,Navas,subdivision,7,4,4,138.40,56.350,246.675,0.2195,...,0.5714,0.4286,0.5714,0.5000,0.5000,0.7500,0.2500,0.2500,tier_3,True
7,barcelona,Pedralbes,subdivision,3,2,2,182.45,158.475,206.425,0.1055,...,0.6667,0.6667,0.6667,0.0000,0.0000,0.5000,0.5000,0.0000,tier_3,False
8,barcelona,Porta,subdivision,2,1,0,43.40,43.400,43.400,0.0890,...,0.0000,0.0000,0.5000,NaN,NaN,NaN,NaN,0.0000,tier_3,False
9,barcelona,Provençals del Poblenou,subdivision,15,8,10,141.25,93.500,201.550,0.1255,...,0.6667,0.3333,0.5333,0.4000,0.5000,0.5000,0.3000,0.0000,tier_3,True


## 4. Reconciliation checks

Confirm the KPI table sums back to the listing-level totals — catches silent aggregation bugs early.

In [5]:
checks = []
for city, sub in features.groupby('city'):
    kpis_city = kpis[kpis['city'] == city]
    located = sub[sub['geo_level'] != 'unknown']
    checks.append({
        'city': city,
        'listings_total': len(sub),
        'listings_located': len(located),
        'kpi_density_sum': int(kpis_city['str_density'].sum()),
        'density_matches': len(located) == int(kpis_city['str_density'].sum()),
        'entire_home_listings': int(located['entire_home_flag'].sum()),
        'kpi_entire_home_sum': int(kpis_city['entire_home_count'].sum()),
        'entire_home_matches': int(located['entire_home_flag'].sum()) == int(kpis_city['entire_home_count'].sum()),
        'breach_90_listings': int(located['breach_90_flag'].sum()),
        'kpi_breach_90_sum': int(kpis_city['breach_count_90'].sum()),
        'breach_90_matches': int(located['breach_90_flag'].sum()) == int(kpis_city['breach_count_90'].sum()),
    })
pd.DataFrame(checks)

,city,listings_total,listings_located,kpi_density_sum,density_matches,entire_home_listings,kpi_entire_home_sum,entire_home_matches,breach_90_listings,kpi_breach_90_sum,breach_90_matches
0,barcelona,2594,2354,2354,True,1414,1414,True,611,611,True
1,london,9643,9599,9599,True,6504,6504,True,1482,1482,True


## 5. Top neighbourhoods per KPI — sanity / preview

These are exactly the kind of rankings Member 4's chatbot will recite when asked "which neighbourhoods have the highest X?"

In [6]:
def show_top(kpi: str, n: int = 10, city: str | None = None) -> pd.DataFrame:
    df = kpis if city is None else kpis[kpis['city'] == city]
    cols = ['city', 'geo_key', 'geo_level', 'str_density', 'entire_home_count', kpi]
    return df.sort_values(kpi, ascending=False).head(n)[cols]

print('Top 10 by STR density — Barcelona:')
display(show_top('str_density', city='barcelona'))
print('\nTop 10 by STR density — London:')
display(show_top('str_density', city='london'))
print('\nTop 10 by 90-night breach count — Barcelona:')
display(show_top('breach_count_90', city='barcelona'))
print('\nTop 10 by 90-night breach count — London:')
display(show_top('breach_count_90', city='london'))
print('\nTop 10 by entire-home share (min 30 listings) — Barcelona:')
display(kpis[(kpis['city']=='barcelona') & (kpis['str_density']>=30)].sort_values('entire_home_share', ascending=False).head(10)[['city','geo_key','str_density','entire_home_count','entire_home_share']])
print('\nTop 10 by entire-home share (min 30 listings) — London:')
display(kpis[(kpis['city']=='london') & (kpis['str_density']>=30)].sort_values('entire_home_share', ascending=False).head(10)[['city','geo_key','str_density','entire_home_count','entire_home_share']])

Top 10 by STR density — Barcelona:


,city,geo_key,geo_level,str_density,entire_home_count,str_density
45,barcelona,la Dreta de l'Eixample,subdivision,295,201,295
39,barcelona,el Raval,subdivision,203,86,203
15,barcelona,"Sant Pere, Santa Caterina i la Ribera",subdivision,163,100,163
2,barcelona,Gothic Quarter,subdivision,160,71,160
41,barcelona,l'Antiga Esquerra de l'Eixample,subdivision,148,94,148
54,barcelona,la Sagrada Família,subdivision,144,106,144
36,barcelona,el Poble-sec,subdivision,118,83,118
52,barcelona,la Nova Esquerra de l'Eixample,subdivision,114,60,114
62,barcelona,la Vila de Gràcia,subdivision,104,76,104
11,barcelona,Sant Antoni,subdivision,79,51,79



Top 10 by STR density — London:


,city,geo_key,geo_level,str_density,entire_home_count,str_density
543,london,Whitechapel,subdivision,242,148,242
540,london,Westbourne Green,subdivision,210,178,210
344,london,Marylebone,neighborhood,178,153,178
183,london,Earl's Court,subdivision,168,145,168
390,london,Paddington,neighborhood,152,128,152
215,london,Fulham,subdivision,139,113,139
535,london,West Kensington,subdivision,127,90,127
143,london,Chelsea,subdivision,123,112,123
384,london,Notting Hill,subdivision,119,106,119
307,london,London Borough of Ealing,neighborhood,117,71,117



Top 10 by 90-night breach count — Barcelona:


...

,city,geo_key,geo_level,str_density,entire_home_count,breach_count_90
543,london,Whitechapel,subdivision,242,148,44
390,london,Paddington,neighborhood,152,128,42
540,london,Westbourne Green,subdivision,210,178,41
344,london,Marylebone,neighborhood,178,153,35
378,london,North Kensington,subdivision,104,82,26
143,london,Chelsea,subdivision,123,112,26
535,london,West Kensington,subdivision,127,90,24
82,london,Barnsbury,subdivision,109,77,24
183,london,Earl's Court,subdivision,168,145,23
446,london,Shoreditch,subdivision,63,54,23



Top 10 by entire-home share (min 30 listings) — Barcelona:


,city,geo_key,str_density,entire_home_count,entire_home_share
63,barcelona,les Corts,32,28,0.8750
27,barcelona,el Camp d'en Grassot i Gràcia Nova,37,29,0.7838
54,barcelona,la Sagrada Família,144,106,0.7361
42,barcelona,la Barceloneta,68,50,0.7353
62,barcelona,la Vila de Gràcia,104,76,0.7308
37,barcelona,el Poblenou,59,43,0.7288
12,barcelona,Sant Gervasi - Galvany,44,32,0.7273
36,barcelona,el Poble-sec,118,83,0.7034
45,barcelona,la Dreta de l'Eixample,295,201,0.6814
11,barcelona,Sant Antoni,79,51,0.6456



Top 10 by entire-home share (min 30 listings) — London:


,city,geo_key,str_density,entire_home_count,entire_home_share
117,london,Brompton,100,95,0.9500
327,london,London Borough of Wandsworth,36,34,0.9444
345,london,Mayfair,33,31,0.9394
143,london,Chelsea,123,112,0.9106
384,london,Notting Hill,119,106,0.8908
454,london,South Kensington,87,76,0.8736
265,london,Holborn,74,64,0.8649
183,london,Earl's Court,168,145,0.8631
344,london,Marylebone,178,153,0.8596
446,london,Shoreditch,63,54,0.8571


## 6. Cross-city comparison summary (Q6)

In [7]:
summary_metrics = ['str_density', 'entire_home_share', 'commercial_host_share',
                   'avg_occupancy', 'median_nightly_price', 'breach_rate_90']

city_summary = kpis.groupby('city')[summary_metrics].agg(['median', 'mean'])
city_summary.round(3)

str_density         entire_home_share        commercial_host_share  \
               median    mean            median   mean                median   
city                                                                           
barcelona        10.0  35.667             0.530  0.489                 0.134   
london            8.0  19.431             0.625  0.589                 0.000   

                 avg_occupancy        median_nightly_price           \
            mean        median   mean               median     mean   
city                                                                  
barcelona  0.153         0.124  0.126               118.85  126.252   
london     0.070         0.078  0.097               142.75  154.160   

          breach_rate_90         
                  median   mean  
city                             
barcelona          0.447  0.406  
london             0.188  0.215

## 7. Save outputs

In [8]:
out_path = PROCESSED_DIR / 'neighbourhood_kpis.csv'
kpis.to_csv(out_path, index=False)
print(f'Saved KPI table -> {out_path}')
print(f'Shape: {kpis.shape}')

dict_path = PROCESSED_DIR / 'data_dictionary_neighbourhood_kpis.csv'
kpi_data_dictionary().to_csv(dict_path, index=False)
print(f'Saved data dictionary -> {dict_path}')

Saved KPI table -> C:\Users\moham\Desktop\Term 3\Capstone\Repo\KPMG_Airbnb_Capstone\data\processed\neighbourhood_kpis.csv


Shape: (560, 27)
Saved data dictionary -> C:\Users\moham\Desktop\Term 3\Capstone\Repo\KPMG_Airbnb_Capstone\data\processed\data_dictionary_neighbourhood_kpis.csv


## 8. Handover note for Member 3

**Input:** `data/processed/neighbourhood_kpis.csv` (this notebook's output)

**What's ready:**
- All neighbourhood KPIs computed, both cities stacked
- `breach_count_90 / 60 / 30` are the policy-simulation answers — already at neighbourhood level
- `reside_unregistered_count` is regulatorily meaningful for Barcelona only (RESIDE proxy)
- All shares already 0–1 normalised — feed directly into KMeans

**What you (M3) need to add:**
- `cluster_label` (saturated / emerging / low-impact) — cluster on the numeric KPI columns
- `risk_priority_score` — weighted 0–100 composite of density, entire_home_share, commercial_host_share, breach_rate_90
- Optionally a price model — separate notebook, not on this table

**To extend in code:**
```python
from src.kpis import compute_neighbourhood_kpis
# pass listing-level features in, get a neighbourhood-level KPI table back
```

**Geo key strategy:**
- `geo_key` is subdivision when present, neighborhood otherwise — group/join on it
- `geo_level` tells you the resolution per row (don't compare a subdivision row to a borough row blindly)